# Práctica de Laboratorio - Sesión 3
## Preprocesamiento de Texto: Tokenización, Stemming y Lematización

**Curso:** NLP y Análisis Semántico  
**Prof.:** Francisco Suárez  
**Semana 1 - Sesión 3**

---

### Objetivos
1. Aplicar diferentes métodos de tokenización sobre texto en español
2. Comparar stemming y lematización de forma práctica
3. Construir un pipeline de preprocesamiento reutilizable
4. Analizar el impacto del preprocesamiento en la representación del texto

### Instrucciones
- Ejecuta cada celda en orden
- Lee los comentarios y completa las secciones marcadas con `# TODO`
- Responde las preguntas de reflexión al final

## Parte 0: Instalación y Configuración

In [ ]:
# Instalar dependencias (ejecutar solo una vez)
# !pip install nltk spacy
# !python -m spacy download es_core_news_sm

In [ ]:
import re
import nltk
import spacy
from collections import Counter

# Descargar recursos de NLTK
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords

# Cargar modelo de spaCy en español
nlp = spacy.load("es_core_news_sm")

print("✅ Todas las librerías cargadas correctamente.")

## Parte 1: Tokenización

Vamos a comparar diferentes métodos de tokenización sobre el mismo texto.

In [ ]:
# Texto de ejemplo: un párrafo con varios desafíos para un tokenizador
texto = """El Dr. García trabaja en la U.C.B. desde 2019. Su correo es 
garcia@ucb.edu.bo y su teléfono es +591-71234567. ¡Hoy presentó un 
proyecto de $2.500,00 sobre NLP! ¿No es increíble? "Sí", dijo él."""

print("Texto de trabajo:")
print(texto)

### 1.1 Tokenización con `split()`

In [ ]:
# Método más simple: dividir por espacios
tokens_split = texto.split()

print(f"Método: split()")
print(f"Cantidad de tokens: {len(tokens_split)}")
print(f"Tokens: {tokens_split}")
print(f"\n⚠️ Problemas detectados:")
print(f"  - 'U.C.B.' incluye el punto final: '{[t for t in tokens_split if 'U.C.B' in t]}'")
print(f"  - 'NLP!' tiene puntuación pegada: '{[t for t in tokens_split if 'NLP' in t]}'")

### 1.2 Tokenización con Regex

In [ ]:
# Regex: separar palabras y puntuación
tokens_regex = re.findall(r"\w+|[^\w\s]", texto)

print(f"Método: regex r'\\w+|[^\\w\\s]'")
print(f"Cantidad de tokens: {len(tokens_regex)}")
print(f"Tokens: {tokens_regex}")

### 1.3 Tokenización con NLTK

In [ ]:
# NLTK: tokenizador entrenado para manejar abreviaciones, etc.
tokens_nltk = word_tokenize(texto, language='spanish')

print(f"Método: NLTK word_tokenize")
print(f"Cantidad de tokens: {len(tokens_nltk)}")
print(f"Tokens: {tokens_nltk}")

# Tokenización por oraciones
oraciones = sent_tokenize(texto, language='spanish')
print(f"\nOraciones detectadas ({len(oraciones)}):")
for i, sent in enumerate(oraciones, 1):
    print(f"  {i}. {sent.strip()}")

### 1.4 Tokenización con spaCy

In [ ]:
# spaCy: tokenización con información lingüística
doc = nlp(texto)
tokens_spacy = [token.text for token in doc]

print(f"Método: spaCy")
print(f"Cantidad de tokens: {len(tokens_spacy)}")
print(f"Tokens: {tokens_spacy}")

# spaCy también detecta oraciones
print(f"\nOraciones detectadas:")
for i, sent in enumerate(doc.sents, 1):
    print(f"  {i}. {sent.text.strip()}")

### 1.5 Tabla comparativa

In [ ]:
# Comparación lado a lado
metodos = {
    "split()": tokens_split,
    "regex": tokens_regex,
    "NLTK": tokens_nltk,
    "spaCy": tokens_spacy
}

print(f"{'Método':<10} {'# Tokens':>10}")
print("-" * 22)
for nombre, tokens in metodos.items():
    print(f"{nombre:<10} {len(tokens):>10}")

# ¿Qué tokens son diferentes entre métodos?
print(f"\n--- Diferencias clave ---")
print(f"¿Cómo maneja cada método 'U.C.B.'?")
for nombre, tokens in metodos.items():
    ucb = [t for t in tokens if 'U' in t or 'C' in t and 'B' in t]
    print(f"  {nombre}: {ucb[:5]}")

### 🏋️ Ejercicio 1

Escribe un tokenizador con regex que maneje correctamente:
- Menciones (`@usuario`)
- Hashtags (`#tema`)
- Emails (`user@mail.com`)
- Palabras normales y puntuación

In [ ]:
tweet = "@maria ¡Excelente clase de #NLP hoy! Escribe a info@ucb.edu.bo para más detalles 🎉"

# TODO: Escribe un patrón regex que capture menciones, hashtags, emails y palabras
# Pista: el orden de las alternativas importa (pon los patrones más específicos primero)
patron = r""  # <-- Completa aquí

tokens = re.findall(patron, tweet)
print(f"Tweet: {tweet}")
print(f"Tokens: {tokens}")

# Resultado esperado debería incluir: '@maria', '#NLP', 'info@ucb.edu.bo' como tokens únicos

## Parte 2: Normalización

Aplicaremos técnicas de normalización paso a paso.

In [ ]:
texto_sucio = """¡¡¡OFERTA INCREÍBLE!!!  Visita https://tienda.com.bo 😍😍
Escríbenos a ventas@tienda.com   o al +591-71234567.
    Precio: $99.99 USD   ¡¡NO TE LO PIERDAS!! #Oferta @tienda_bo"""

print("Texto original:")
print(texto_sucio)

### 2.1 Normalización paso a paso

In [ ]:
texto_proc = texto_sucio

# Paso 1: Convertir a minúsculas
texto_proc = texto_proc.lower()
print(f"1. Lowercase:\n   {texto_proc}\n")

# Paso 2: Eliminar URLs
texto_proc = re.sub(r"https?://\S+", "", texto_proc)
print(f"2. Sin URLs:\n   {texto_proc}\n")

# Paso 3: Eliminar emails
texto_proc = re.sub(r"\S+@\S+", "", texto_proc)
print(f"3. Sin emails:\n   {texto_proc}\n")

# Paso 4: Eliminar menciones y hashtags
texto_proc = re.sub(r"[@#]\w+", "", texto_proc)
print(f"4. Sin @/# :\n   {texto_proc}\n")

# Paso 5: Eliminar emojis y caracteres especiales
texto_proc = re.sub(r"[^\w\sáéíóúñü¿?¡!.,]", "", texto_proc)
print(f"5. Sin emojis:\n   {texto_proc}\n")

# Paso 6: Normalizar puntuación repetida
texto_proc = re.sub(r"([¿?¡!.,])\1+", r"\1", texto_proc)
print(f"6. Punt. normalizada:\n   {texto_proc}\n")

# Paso 7: Normalizar espacios
texto_proc = re.sub(r"\s+", " ", texto_proc).strip()
print(f"7. Espacios normalizados:\n   {texto_proc}")

### 2.2 Stopwords

In [ ]:
# Explorar las stopwords en español
stops_es = set(stopwords.words('spanish'))

print(f"Total de stopwords en español: {len(stops_es)}")
print(f"\nPrimeras 30 (ordenadas):")
print(sorted(stops_es)[:30])

# Aplicar eliminación de stopwords
oracion = "el procesamiento de lenguaje natural es una de las ramas más importantes de la inteligencia artificial"
tokens = word_tokenize(oracion, language='spanish')
tokens_sin_stops = [t for t in tokens if t not in stops_es]

print(f"\nOriginal ({len(tokens)} tokens): {tokens}")
print(f"Sin stops ({len(tokens_sin_stops)} tokens): {tokens_sin_stops}")
print(f"Reducción: {len(tokens) - len(tokens_sin_stops)} tokens eliminados ({(len(tokens) - len(tokens_sin_stops))/len(tokens)*100:.0f}%)")

## Parte 3: Stemming

Usaremos el **Snowball Stemmer** de NLTK, que soporta español.

In [ ]:
stemmer = SnowballStemmer("spanish")

# Familias de palabras
familias = {
    "correr": ["correr", "corriendo", "corrió", "corredor", "corrida", "corredores"],
    "computar": ["computar", "computadora", "computadoras", "computación", "computacional"],
    "estudiar": ["estudiar", "estudiante", "estudiantes", "estudio", "estudios", "estudioso"],
}

for raiz, palabras in familias.items():
    print(f"\n📌 Familia '{raiz}':")
    for palabra in palabras:
        stem = stemmer.stem(palabra)
        print(f"   {palabra:<18} → {stem}")

### 3.1 Errores del Stemming

In [ ]:
# Over-stemming: palabras NO relacionadas que producen el mismo stem
print("🔴 OVER-STEMMING (falsos positivos)")
print("Palabras no relacionadas → mismo stem\n")

pares_over = [
    ("universo", "universidad"),
    ("general", "generoso"),
    ("medicina", "medida"),
    ("temporal", "temprano"),
]

for p1, p2 in pares_over:
    s1, s2 = stemmer.stem(p1), stemmer.stem(p2)
    icono = "⚠️" if s1 == s2 else "✅"
    print(f"  {icono} '{p1}' → '{s1}' | '{p2}' → '{s2}' | Iguales: {s1 == s2}")

# Under-stemming: palabras SÍ relacionadas que producen stems diferentes
print(f"\n🔵 UNDER-STEMMING (falsos negativos)")
print("Palabras relacionadas → stems diferentes\n")

pares_under = [
    ("absorber", "absorción"),
    ("producir", "producto"),
    ("decir", "dicho"),
    ("hacer", "hecho"),
]

for p1, p2 in pares_under:
    s1, s2 = stemmer.stem(p1), stemmer.stem(p2)
    icono = "⚠️" if s1 != s2 else "✅"
    print(f"  {icono} '{p1}' → '{s1}' | '{p2}' → '{s2}' | Iguales: {s1 == s2}")

## Parte 4: Lematización

Usaremos **spaCy** para lematizar texto en español.

In [ ]:
texto = "Los mejores estudiantes corrieron rápidamente hacia las universidades más reconocidas del país"
doc = nlp(texto)

print(f"{'Token':<18} {'Lema':<16} {'POS':<8} {'Detalle POS':<12}")
print("-" * 56)
for token in doc:
    print(f"{token.text:<18} {token.lemma_:<16} {token.pos_:<8} {token.tag_:<12}")

### 4.1 Stemming vs. Lematización: Comparación directa

In [ ]:
palabras_test = [
    "corriendo", "mejores", "ciudades", "producción", 
    "dificultades", "haciendo", "fue", "comieron",
    "universidades", "rápidamente", "increíble", "estudiantes"
]

print(f"{'Palabra':<18} {'Stem':<16} {'Lema (spaCy)':<16} {'¿Diferentes?'}")
print("-" * 62)
for palabra in palabras_test:
    stem = stemmer.stem(palabra)
    doc = nlp(palabra)
    lema = doc[0].lemma_
    diff = "🔸 Sí" if stem != lema else ""
    print(f"{palabra:<18} {stem:<16} {lema:<16} {diff}")

### 🏋️ Ejercicio 2

Encuentra **3 palabras en español** donde:
1. El stemming produce un resultado incorrecto pero la lematización es correcta
2. Ambos producen el mismo resultado

In [ ]:
# TODO: Prueba con tus propias palabras
mis_palabras = []  # <-- Agrega al menos 6 palabras aquí

for palabra in mis_palabras:
    stem = stemmer.stem(palabra)
    doc = nlp(palabra)
    lema = doc[0].lemma_
    print(f"{palabra:<18} stem: {stem:<16} lema: {lema:<16}")

## Parte 5: Pipeline Completo de Preprocesamiento

Construyamos una función reutilizable que integre todos los pasos.

In [ ]:
stops = set(stopwords.words('spanish'))

def preprocesar(texto, usar_lemas=True, quitar_stops=True, solo_alfa=True):
    """
    Pipeline de preprocesamiento para texto en español.
    
    Parámetros:
    -----------
    texto : str
        Texto crudo de entrada
    usar_lemas : bool
        Si True, aplica lematización. Si False, devuelve tokens sin lematizar.
    quitar_stops : bool
        Si True, elimina stopwords del español.
    solo_alfa : bool
        Si True, solo mantiene tokens alfabéticos.
    
    Retorna:
    --------
    list[str] : Lista de tokens preprocesados
    """
    # 1. Lowercase
    texto = texto.lower()
    # 2. Eliminar URLs
    texto = re.sub(r"https?://\S+|www\.\S+", "", texto)
    # 3. Eliminar emails
    texto = re.sub(r"\S+@\S+", "", texto)
    # 4. Eliminar menciones y hashtags
    texto = re.sub(r"[@#]\w+", "", texto)
    # 5. Eliminar caracteres no alfanuméricos (mantener acentos)
    texto = re.sub(r"[^\w\sáéíóúñü]", " ", texto)
    # 6. Normalizar espacios
    texto = re.sub(r"\s+", " ", texto).strip()
    # 7. Tokenizar y lematizar con spaCy
    doc = nlp(texto)
    tokens = []
    for token in doc:
        palabra = token.lemma_ if usar_lemas else token.text
        if quitar_stops and palabra in stops:
            continue
        if solo_alfa and not palabra.isalpha():
            continue
        tokens.append(palabra)
    return tokens

print("✅ Función preprocesar() definida correctamente.")

In [ ]:
# Probar el pipeline con diferentes configuraciones
texto_prueba = """Los investigadores de la Universidad Católica Boliviana están 
desarrollando modelos de NLP para idiomas nativos. Visita https://ucb.edu.bo 
o escribe a info@ucb.edu.bo para más información. #IA #NLP @UCB_Bolivia"""

print(f"TEXTO ORIGINAL:\n{texto_prueba}\n")
print("=" * 60)

# Configuración 1: Con lemas, sin stopwords
r1 = preprocesar(texto_prueba, usar_lemas=True, quitar_stops=True)
print(f"\n1️⃣ Lemas + sin stops ({len(r1)} tokens):")
print(f"   {r1}\n")

# Configuración 2: Sin lemas, sin stopwords
r2 = preprocesar(texto_prueba, usar_lemas=False, quitar_stops=True)
print(f"2️⃣ Sin lemas + sin stops ({len(r2)} tokens):")
print(f"   {r2}\n")

# Configuración 3: Con lemas, con stopwords
r3 = preprocesar(texto_prueba, usar_lemas=True, quitar_stops=False)
print(f"3️⃣ Lemas + con stops ({len(r3)} tokens):")
print(f"   {r3}")

## Parte 6: Análisis de Impacto

Veamos cómo el preprocesamiento afecta la representación del texto.

In [ ]:
# Corpus de ejemplo: 5 oraciones sobre NLP
corpus = [
    "El procesamiento de lenguaje natural permite analizar textos de forma automática.",
    "Las redes neuronales profundas han mejorado significativamente el procesamiento del lenguaje.",
    "Los modelos de lenguaje modernos pueden generar textos coherentes y naturales.",
    "La inteligencia artificial está transformando la manera en que procesamos información textual.",
    "Analizar el sentimiento de los textos es una aplicación común del procesamiento natural.",
]

# Preprocesar todo el corpus
corpus_procesado = [preprocesar(doc) for doc in corpus]

print("CORPUS PREPROCESADO:\n")
for i, (orig, proc) in enumerate(zip(corpus, corpus_procesado), 1):
    print(f"Doc {i} (original): {orig[:60]}...")
    print(f"Doc {i} (procesado): {proc}\n")

In [ ]:
# Análisis de vocabulario: antes y después
tokens_crudos = []
for doc in corpus:
    tokens_crudos.extend(word_tokenize(doc.lower(), language='spanish'))

tokens_procesados = []
for doc in corpus_procesado:
    tokens_procesados.extend(doc)

vocab_crudo = set(tokens_crudos)
vocab_procesado = set(tokens_procesados)

print(f"{'Métrica':<30} {'Crudo':>10} {'Procesado':>12} {'Reducción':>12}")
print("-" * 66)
print(f"{'Total de tokens':<30} {len(tokens_crudos):>10} {len(tokens_procesados):>12} {len(tokens_crudos)-len(tokens_procesados):>10} ({(1-len(tokens_procesados)/len(tokens_crudos))*100:.0f}%)")
print(f"{'Vocabulario (tipos únicos)':<30} {len(vocab_crudo):>10} {len(vocab_procesado):>12} {len(vocab_crudo)-len(vocab_procesado):>10} ({(1-len(vocab_procesado)/len(vocab_crudo))*100:.0f}%)")

# Palabras más frecuentes después del procesamiento
freq = Counter(tokens_procesados)
print(f"\n📊 Top 10 palabras más frecuentes (procesado):")
for palabra, cuenta in freq.most_common(10):
    print(f"   '{palabra}': {cuenta}")

## 🏋️ Ejercicio Final: Preprocesador de Noticias

Aplica el pipeline completo a los siguientes titulares de noticias y responde las preguntas.

In [ ]:
noticias = [
    "¡ÚLTIMA HORA! El gobierno boliviano invertirá $500 millones en tecnología e innovación.",
    "Investigadores de la UMSA desarrollaron un traductor automático para lenguas originarias.",
    "Las universidades privadas están implementando programas de inteligencia artificial.",
    "Bolivia apuesta por la transformación digital: nuevos centros tecnológicos en 3 ciudades.",
    "Estudiantes universitarios presentaron proyectos innovadores en la feria de ciencias 2026."
]

# TODO: Preprocesa cada noticia y guarda los resultados
# Pista: usa la función preprocesar() que definimos arriba
noticias_procesadas = []  # <-- Completa aquí

# TODO: Calcula el vocabulario total del corpus preprocesado
# vocabulario = ???

# TODO: Encuentra las 5 palabras más frecuentes
# frecuencias = ???

## Preguntas de Reflexión

Responde en una celda de texto (doble click para editar):

1. **¿En qué caso usarías stemming en lugar de lematización?** Menciona un escenario concreto.

2. **¿Siempre debemos eliminar las stopwords?** Da un ejemplo donde eliminarlas sería perjudicial.

3. **¿Qué pasa con palabras como "no" en un análisis de sentimiento?** ¿Es stopword o no?

4. **¿Qué desafíos específicos tiene el preprocesamiento de texto en español** comparado con el inglés?

*Escribe tus respuestas aquí:*

1. ...
2. ...
3. ...
4. ...